In [1]:
# ======================================================================
# CLEAN DUBLIN PLANNING: one ML-ready table, one row per application
# ======================================================================
# Input:  just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv
# Output: just_dublin_dataset\Just_dublin_clean_dataset\planning\planning_dublin_clean.csv
#
# Same cleaning as the national planning notebook, for the 4 Dublin
# councils, keeping the location columns:
#   nation, province, county, local_authority (the council), dublin_district
#
# What this table is for (Student 2):
#   - TF-IDF / n-grams + LDA topics      -> text: DevelopmentDescription
#   - ElasticNet regression              -> target: days_to_decision
#   - Linear SVM classification          -> target: is_refused
#
# Rules: fix the data, do NOT fill missing values / scale / encode here,
# and leave out anything recorded after the decision (leakage).
#
# Needs the Dublin processed notebooks (00 and 02) to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

planning_file = base + r'\just_dublin_dataset\just_dublin_processed_dataset\planning\planning_dublin.csv'
dublin_clean  = base + r'\just_dublin_dataset\Just_dublin_clean_dataset'

In [2]:
planning = pd.read_csv(planning_file, low_memory=False)

print(planning.shape)
planning.head(3)

(43351, 42)


,OBJECTID,PlanningAuthority,ApplicationNumber,DevelopmentDescription,DevelopmentAddress,DevelopmentPostcode,ApplicationStatus,ApplicationType,Decision,LandUseCode,AreaofSite,NumResidentialUnits,OneOffHouse,FloorArea,ReceivedDate,WithdrawnDate,DecisionDate,DecisionDueDate,GrantDate,ExpiryDate,AppealRefNumber,AppealStatus,AppealDecision,AppealDecisionDate,AppealSubmittedDate,FIRequestDate,FIRecDate,LinkAppDetails,OneOffKPI,ETL_DATE,SiteId,word_count,county,year,quarter,days_to_decision,local_authority,dublin_district,district_id,county_id,province,nation
0,178001,Dublin City Council,3605/23,RETENTION Permission for the following : \r\n(a) new shopfront with\r\n(b) back-lit sign on fascia\r\n(c) new illumi...,"146 Baggot Street Lower, Dublin 2, D02 T990",NaN,Decision Notice Issued,Retention Permission,GRANT RETENTION PERMISSIO,NaN,40.0,NaN,NaN,NaN,2023-04-21,NaN,2023-06-14,2023-06-15,2023-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,36,Dublin,2023,2023Q2,54.0,Dublin City Council,Dublin 2,2,6,Leinster,Ireland
1,178003,Dublin City Council,3631/23,The development will consist of the construction of new fencing and the access controlled gate access to the apartme...,"98-100 South Circular Road, Dublin 8, D08 V5YA",NaN,Decision Notice Issued,Permission,GRANT PERMISSION,NaN,35.0,NaN,NaN,NaN,2023-04-23,NaN,2023-06-19,2023-06-19,2023-08-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,34,Dublin,2023,2023Q2,57.0,Dublin City Council,Dublin 8,9,6,Leinster,Ireland
2,178004,Dublin City Council,WEB1524/24,"Planning permission for attic conversion to use as study/storage space, replacing 2 No. existing roof windows with 2...","7, Church Park Court, Dublin 6w, D6W DW65",NaN,Decision Notice Issued,Permission,APPLICATION DECLARED INVA,NaN,32.0,NaN,NaN,NaN,2024-04-30,NaN,2024-05-02,2024-06-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-31,NaN,37,Dublin,2024,2024Q2,2.0,Dublin City Council,Dublin 6W,7,6,Leinster,Ireland


In [3]:
# Removing extra spaces from all text. Different councils type the same
# value with spaces at the end ('GRANT PERMISSION   '), which made the
# same label look like 3 different ones.

text_cols = []
for col in planning.columns:
    if planning[col].dtype == 'object' or str(planning[col].dtype) == 'str':
        text_cols.append(col)

for col in text_cols:
    planning[col] = planning[col].str.strip()
    planning[col] = planning[col].replace('', np.nan)

print('Text columns cleaned:', len(text_cols))

Text columns cleaned: 31


In [4]:
# Duplicates: some applications appear twice with everything the same except
# OBJECTID (the ArcGIS row number). Those are the same application twice.

cols_without_id = []
for col in planning.columns:
    if col != 'OBJECTID':
        cols_without_id.append(col)

print('Exact duplicates (ignoring OBJECTID):', planning.duplicated(subset=cols_without_id).sum())

rows_before = len(planning)
planning = planning.drop_duplicates(subset=cols_without_id)
print('Removed', rows_before - len(planning), 'rows. Rows left:', len(planning))

Exact duplicates (ignoring OBJECTID): 139
Removed 139 rows. Rows left: 43212


In [5]:
# Dates come back from CSV as text, so converting again.

for col in ['ReceivedDate', 'DecisionDate']:
    planning[col] = pd.to_datetime(planning[col], errors='coerce')

print('Received from', planning['ReceivedDate'].min(), 'to', planning['ReceivedDate'].max())
print('Latest decision date:', planning['DecisionDate'].max())

Received from 2021-01-04 00:00:00 to 2026-10-13 00:00:00
Latest decision date: 2026-10-13 00:00:00


In [6]:
# Location check: every application should have a council and a district
# (a real one, or 'Dublin (no district)').

print('Rows without a district:', planning['dublin_district'].isnull().sum())   # should be 0
print('Rows without a council: ', planning['local_authority'].isnull().sum())   # should be 0
print()
print(planning['local_authority'].value_counts())

Rows without a district: 0
Rows without a council:  0

local_authority
Dublin City Council                      18268
Dun Laoghaire Rathdown County Council     9707
Fingal County Council                     8901
South Dublin County Council               6336
Name: count, dtype: int64


In [7]:
# The Decision column is very messy: about 180 different spellings for a
# handful of real outcomes, e.g.
#   'CONDITIONAL', 'GRANT PERMISSION', 'Granted (Conditional)', 'Grant with Conditions'
# Looking at the most common ones first.

planning['Decision'].value_counts(dropna=False).head(40)

Decision
GRANT PERMISSION                             19677
APPLICATION DECLARED INVA                     4328
Grant Permission                              3758
REFUSE PERMISSION                             2820
APPLICATION DECLARED INVALID                  1414
GRANT PERMISSION FOR RETENTION                1017
NaN                                            965
Invalid Application                            924
DECLARE APPLICATION INVALID                    799
GRANT RETENTION PERMISSIO                      742
GRANT PERMISSION & GRANT RETENTION             529
Refuse Permission                              363
Grant Permission for Retention                 341
GRANT PERMISSION AND RETE                      330
REFUSE PERMISSION FOR RETENTION                300
INVALID APPLICATION DUE TO SITE NOTICE         287
Declared Exempt                                254
S5 DEC EXEMPT                                  239
Grant Permission & Grant Retention             194
GRANT RETENTION PERMIS

In [8]:
# Grouping all decision labels into a few clear outcomes by looking for key
# words. The order matters (e.g. 'App withdrawn as no AI recd' must be
# Withdrawn, and 'GRANT CERTIFICATE OF EXEMPTION' is not a normal grant).
#
#   Pending    -> no decision yet, or still waiting for further information
#   Invalid    -> application declared invalid
#   Withdrawn  -> applicant withdrew / deemed withdrawn
#   Exempt     -> Section 5 / exemption decisions (not a normal planning decision)
#   Split      -> part granted and part refused
#   Refused
#   Granted    -> conditional or unconditional permission / approval
#   Other      -> anything else (quashed by court, decided by another body...)

def decision_group(text):
    if pd.isnull(text):
        return 'Pending'

    t = text.upper()

    if 'INVA' in t:                     # some labels are cut short: 'APPLICATION DECLARED INVA'
        return 'Invalid'
    if 'WITHDRAW' in t:
        return 'Withdrawn'
    if 'EXEMPT' in t or t.startswith('S5') or 'SECTION 5' in t or 'S179A' in t:
        return 'Exempt'
    if 'SPLIT' in t or ('GRANT' in t and ('REFUSE' in t or 'REF.' in t or 'REF ' in t)):
        return 'Split'
    if ('INFORMATION' in t or 'REQ AI' in t or 'REQUEST AI' in t or 'CLARIFICATION' in t
            or 'EXTENSION OF TIME' in t or 'TIME EXT' in t):
        return 'Pending'
    if 'REFUSE' in t or 'REJECTED' in t:
        return 'Refused'
    if 'GRANT' in t or 'CONDITIONAL' in t or 'APPROV' in t:
        return 'Granted'
    return 'Other'


planning['decision_group'] = planning['Decision'].apply(decision_group)

print(planning['decision_group'].value_counts())

decision_group
Granted      27141
Invalid       8292
Refused       4060
Pending       1351
Exempt         994
Withdrawn      923
Split          421
Other           30
Name: count, dtype: int64


In [9]:
# Checking the grouping: every raw label next to the group it went into.
# Worth scrolling through to make sure nothing landed in the wrong place.

check = planning.groupby(['decision_group', 'Decision']).size().reset_index(name='count')
check = check.sort_values(['decision_group', 'count'], ascending=[True, False])
check

,decision_group,Decision,count
1,Exempt,Declared Exempt,254
11,Exempt,S5 DEC EXEMPT,239
4,Exempt,Declared Not Exempt,142
8,Exempt,GRANT CERTIFICATE OF EXEMPTION,134
12,Exempt,S5 DEC NOT EXEMPT,101
7,Exempt,GRANT CERT OF EXEMPTION (<=4 unit),33
14,Exempt,S5 SPLIT DECISION,32
13,Exempt,S5 REQ AI,20
2,Exempt,Declared Exempt & Declared Not Exempt,17
6,Exempt,GRANT CERT OF EXEMPTION (0.1 Hec ),12


In [10]:
# Classification target for the Linear SVM: refused (1) vs granted (0).
# Only applications with a clear Granted or Refused outcome get a label.
# Everything else (pending, invalid, withdrawn, split, exempt) stays empty,
# so it can still be used for the text mining but not for this model.

planning['is_refused'] = np.nan
planning.loc[planning['decision_group'] == 'Granted', 'is_refused'] = 0
planning.loc[planning['decision_group'] == 'Refused', 'is_refused'] = 1

print(planning['is_refused'].value_counts(dropna=False))
print()
print('Refusal rate among decided:', round(planning['is_refused'].mean(), 3))

is_refused
0.0    27141
NaN    12011
1.0     4060
Name: count, dtype: int64

Refusal rate among decided: 0.13


In [11]:
# Regression target for ElasticNet: days from received date to decision date.
# Problems found:
#   - some decision dates are BEFORE the received date (negative days)
#   - one decision date is in 2075
# Those can't be real, so they become missing.
# Also only keeping days for real decisions (Granted / Refused / Split); an
# 'invalid' or 'withdrawn' date is not a decision on the application.

planning['days_to_decision'] = (planning['DecisionDate'] - planning['ReceivedDate']).dt.days

print('Negative days:', (planning['days_to_decision'] < 0).sum())
print('Decision dates after Oct 2026:', (planning['DecisionDate'] > '2026-10-31').sum())

planning.loc[planning['days_to_decision'] < 0, 'days_to_decision'] = np.nan
planning.loc[planning['DecisionDate'] > '2026-10-31', 'days_to_decision'] = np.nan
planning.loc[~planning['decision_group'].isin(['Granted', 'Refused', 'Split']), 'days_to_decision'] = np.nan

print(planning['days_to_decision'].describe().round(1))

Negative days: 8
Decision dates after Oct 2026: 0
count    31612.0
mean        73.1
std         58.7
min          0.0
25%         51.0
50%         55.0
75%         62.0
max       1977.0
Name: days_to_decision, dtype: float64


In [12]:
# Application type: same problem as Decision (PERMISSION, Permission,
# 'Permission   '...). Grouping into a few clear types.

def application_group(text):
    if pd.isnull(text):
        return np.nan

    t = text.upper()

    if 'CONSEQ' in t or 'ON FOOT' in t or 'FOLLOWING GRANT' in t:
        return 'Consequent on outline'
    if 'EXTENSION' in t:
        return 'Extension of duration'
    if 'EXEMPT' in t or 'SECTION 5' in t or 'SECT. 5' in t or '179' in t:
        return 'Exemption'
    if 'SHD' in t or 'LRD' in t or 'SDZ' in t or 'STRATEGIC' in t:
        return 'Large scale (SHD/LRD/SDZ)'
    if 'PART 8' in t or 'PART VIII' in t or 'PART VLLL' in t or 'PART X' in t or 'PART 10' in t:
        return 'Council own development'
    if 'OUTLINE' in t:
        return 'Outline permission'
    if 'RETENTION' in t and 'PERMISSION AND' in t:
        return 'Permission and retention'
    if 'RETENTION' in t:
        return 'Retention'
    if 'PERMISSION' in t:
        return 'Permission'
    return 'Other'


planning['application_group'] = planning['ApplicationType'].apply(application_group)

print(planning['application_group'].value_counts(dropna=False))

application_group
Permission                   35847
Retention                     4419
Exemption                     1088
Permission and retention       851
Large scale (SHD/LRD/SDZ)      491
Extension of duration          266
Council own development        106
Outline permission              88
Other                           35
Consequent on outline           21
Name: count, dtype: int64


In [13]:
# One-off house: written as Y, Yes, One, Single House, No... -> 1 / 0.
# Blank stays missing.

print(planning['OneOffHouse'].value_counts(dropna=False))

one_off_map = {'Y': 1, 'Yes': 1, 'One': 1, 'Single House': 1, 'No': 0, 'N': 0}
planning['one_off_house'] = planning['OneOffHouse'].map(one_off_map)

print(planning['one_off_house'].value_counts(dropna=False))

OneOffHouse
NaN             35882
No               6022
Single House      994
Yes               314
Name: count, dtype: int64
one_off_house
NaN    35882
0.0     6022
1.0     1308
Name: count, dtype: int64


In [14]:
# Text column: keeping the original description, and adding a lower-case
# version with extra spaces removed (easier for TF-IDF / LDA later).
# Stop words, n-grams etc. are left for the text-mining notebook.

planning['clean_text'] = planning['DevelopmentDescription'].fillna('').str.lower()
planning['clean_text'] = planning['clean_text'].str.split().str.join(' ')

planning['word_count'] = planning['clean_text'].str.split().str.len()

print('Average words:', round(planning['word_count'].mean(), 1))
print('Median words: ', planning['word_count'].median())
print('Empty descriptions:', (planning['word_count'] == 0).sum())

Average words: 45.1
Median words:  20.0
Empty descriptions: 0


In [15]:
# Residential flag (the proposal mentions residential-project filters).
# An application counts as residential if it has residential units, is a
# one-off house, or the description mentions a home.

home_words = ['dwelling', 'house', 'apartment', 'residential', 'housing', 'home']

mentions_home = planning['clean_text'].str.contains('|'.join(home_words))

planning['is_residential'] = 0
planning.loc[planning['NumResidentialUnits'] > 0, 'is_residential'] = 1
planning.loc[planning['one_off_house'] == 1, 'is_residential'] = 1
planning.loc[mentions_home, 'is_residential'] = 1

print(planning['is_residential'].value_counts(normalize=True).round(3))

is_residential
0    0.667
1    0.333
Name: proportion, dtype: float64


In [16]:
# Quarter and year from the received date (same format as BER, PPR, rent).

planning['year'] = planning['ReceivedDate'].dt.year
planning['quarter'] = planning['ReceivedDate'].dt.to_period('Q').astype(str)

planning['year'].value_counts().sort_index()

year
2021    7744
2022    7481
2023    6904
2024    7740
2025    8068
2026    5275
Name: count, dtype: int64

In [17]:
# Numeric columns: floor area and site area have some huge values
# (site area up to 10 billion). Not deleting them here, just looking.
# Outliers are handled at the modelling stage.

planning[['NumResidentialUnits', 'FloorArea', 'AreaofSite']].describe().round(1)

,NumResidentialUnits,FloorArea,AreaofSite
count,10541.0,13952.0,4.301800e+04
mean,3.2,301.9,3.153347e+05
std,35.2,3319.4,5.358028e+07
min,0.0,0.0,0.000000e+00
25%,0.0,0.0,2.000000e+02
50%,0.0,0.0,3.500000e+02
75%,0.0,25.0,8.199000e+02
max,1126.0,147192.0,1.096980e+10


In [18]:
# Choosing the columns for the clean table.
# Left out on purpose:
#   GrantDate, ExpiryDate, all Appeal columns, WithdrawnDate -> after the decision (leakage)
#   ApplicationStatus      -> describes where the case ended up (leakage)
#   FIRequestDate, FIRecDate -> only known while the case is being processed
#   DevelopmentAddress, DevelopmentPostcode -> address details; the district is already
#                         taken from them, so not needed (data minimisation)
#   LandUseCode, OneOffKPI, LinkAppDetails, SiteId, ETL_DATE, DecisionDueDate -> mostly empty or admin
#   Decision, ApplicationType, OneOffHouse -> replaced by the cleaned versions

keep = ['OBJECTID', 'nation', 'province', 'county', 'county_id', 'local_authority',
        'dublin_district', 'district_id', 'PlanningAuthority', 'ApplicationNumber',
        'ReceivedDate', 'DecisionDate', 'year', 'quarter',
        'application_group', 'one_off_house', 'NumResidentialUnits', 'FloorArea', 'AreaofSite',
        'is_residential',
        'DevelopmentDescription', 'clean_text', 'word_count',
        'decision_group', 'is_refused', 'days_to_decision']

planning_clean = planning[keep].rename(columns={'OBJECTID': 'application_id'})

print(planning_clean.shape)
planning_clean.head()

(43212, 26)


,application_id,nation,province,county,county_id,local_authority,dublin_district,district_id,PlanningAuthority,ApplicationNumber,ReceivedDate,DecisionDate,year,quarter,application_group,one_off_house,NumResidentialUnits,FloorArea,AreaofSite,is_residential,DevelopmentDescription,clean_text,word_count,decision_group,is_refused,days_to_decision
0,178001,Ireland,Leinster,Dublin,6,Dublin City Council,Dublin 2,2,Dublin City Council,3605/23,2023-04-21,2023-06-14,2023,2023Q2,Retention,NaN,NaN,NaN,40.0,0,RETENTION Permission for the following : \r\n(a) new shopfront with\r\n(b) back-lit sign on fascia\r\n(c) new illumi...,retention permission for the following : (a) new shopfront with (b) back-lit sign on fascia (c) new illuminated proj...,36,Granted,0.0,54.0
1,178003,Ireland,Leinster,Dublin,6,Dublin City Council,Dublin 8,9,Dublin City Council,3631/23,2023-04-23,2023-06-19,2023,2023Q2,Permission,NaN,NaN,NaN,35.0,1,The development will consist of the construction of new fencing and the access controlled gate access to the apartme...,the development will consist of the construction of new fencing and the access controlled gate access to the apartme...,34,Granted,0.0,57.0
2,178004,Ireland,Leinster,Dublin,6,Dublin City Council,Dublin 6W,7,Dublin City Council,WEB1524/24,2024-04-30,2024-05-02,2024,2024Q2,Permission,NaN,NaN,NaN,32.0,0,"Planning permission for attic conversion to use as study/storage space, replacing 2 No. existing roof windows with 2...","planning permission for attic conversion to use as study/storage space, replacing 2 no. existing roof windows with 2...",37,Invalid,NaN,NaN
3,178005,Ireland,Leinster,Dublin,6,Dublin City Council,Dublin 15,16,Dublin City Council,WEB1297/24,2024-03-15,2024-10-07,2024,2024Q1,Permission,NaN,NaN,NaN,132.7,1,"The Development will consist of the construction of a ground floor, flat roof extension to the rear of the house, co...","the development will consist of the construction of a ground floor, flat roof extension to the rear of the house, co...",88,Granted,0.0,206.0
4,178006,Ireland,Leinster,Dublin,6,Dublin City Council,Dublin 2,2,Dublin City Council,WEB1960/23,2023-11-03,2024-01-03,2023,2023Q4,Permission,NaN,NaN,NaN,1.5,0,The replacement of the existing advertising display (3.25m wide x 3.25m high) with a digital advertising display (2....,the replacement of the existing advertising display (3.25m wide x 3.25m high) with a digital advertising display (2....,78,Refused,1.0,61.0


In [19]:
# Missing values left (% per column). Not filling them here.

missing = (planning_clean.isnull().mean() * 100).round(1)
missing[missing > 0].sort_values(ascending=False)

one_off_house          83.0
NumResidentialUnits    75.6
FloorArea              67.7
is_refused             27.8
days_to_decision       26.8
DecisionDate            2.4
AreaofSite              0.4
dtype: float64

In [20]:
planning_clean.info()

<class 'pandas.DataFrame'>
Index: 43212 entries, 0 to 43350
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   application_id          43212 non-null  int64         
 1   nation                  43212 non-null  str           
 2   province                43212 non-null  str           
 3   county                  43212 non-null  str           
 4   county_id               43212 non-null  int64         
 5   local_authority         43212 non-null  str           
 6   dublin_district         43212 non-null  str           
 7   district_id             43212 non-null  int64         
 8   PlanningAuthority       43212 non-null  str           
 9   ApplicationNumber       43212 non-null  str           
 10  ReceivedDate            43212 non-null  datetime64[us]
 11  DecisionDate            42154 non-null  datetime64[us]
 12  year                    43212 non-null  int32         
 13  qu

In [21]:
# Saving the clean Dublin planning table.

save_folder = dublin_clean + r'\planning'
os.makedirs(save_folder, exist_ok=True)

planning_clean.to_csv(save_folder + r'\planning_dublin_clean.csv', index=False)
print('Saved', planning_clean.shape, 'to', save_folder + r'\planning_dublin_clean.csv')

Saved (43212, 26) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\just_dublin_dataset\Just_dublin_clean_dataset\planning\planning_dublin_clean.csv
